In [1]:
import numpy as np
from phasepy import component, mixture, rkseos
from phasepy.equilibrium import flash, tpd_min
water = component(name='water', Tc=647.13, Pc=220.55, Zc=0.229, Vc=55.948, w=0.344861)
ethanol = component(name='ethanol', Tc=514.0, Pc=61.37, Zc=0.241, Vc=168.0, w=0.643558)
co2 = component(name='co2', Tc=304.1, Pc=73.8, Zc=0.274, Vc=94.0, w=0.225)
co = component(name='co', Tc=132.85, Pc=34.0, Zc=0.274, Vc=94.0, w=0.225)
h2 = component(name='h2', Tc=33.19, Pc=12.97, Zc=0.291, Vc=58.0, w=0.344861)
h2o = component(name='h2o', Tc=647.13, Pc=220.55, Zc=0.229, Vc=55.948, w=0.344861)
mix = mixture(water, ethanol)
mix.add_component(co2)
mix.add_component(co)
mix.add_component(h2)
mix.add_component(h2o)
eos = rkseos(mix, 'qmr')
T = 488.5
P = 50.01
Z = np.array([0, 0, 0.2, 0.0, 0.8, 0.0])
x0 = np.array([0.1, 0.9, 0.0, 0.0, 0.0, 0.0])
y0 = np.array([0.2, 0.8, 0.0, 0.0, 0.0, 0.0])
flash(x0, y0, 'LV', Z, T, P, eos, K_tol=1e-10) # phase compositions, vapor phase fraction

(array([0. , 0. , 0.2, 0. , 0.8, 0. ]),
 array([0. , 0. , 0.2, 0. , 0.8, 0. ]),
 1.0)

In [2]:
from TKA_Mo_fug_coeffs_MeOH import phi_Soave_3

In [3]:
z = np.zeros(12)
z.fill(1)

z[5] = 10

z[:6] = z[:6] / np.sum(z[:6])
z[6:] = z[6:] / np.sum(z[6:])

In [4]:
phi_Soave_3(z, 500 + 273.15, 1e5)

Z 1.0003524777492392 1.0001912391234231


(array([1.00022977, 1.00041138, 0.99952399, 1.0005407 , 0.99989551,
        1.00054582]),
 array([1.00027434, 1.00029277, 0.99983022, 1.00042708, 1.00020516,
        1.0004258 ]))

In [5]:
Z_liq, Z_vap = phi_Soave_3(z, 500 + 273.15, 1e5)

Z 1.0003524777492392 1.0001912391234231


In [6]:
# filter out real roots
Z_liq = Z_liq[np.isreal(Z_liq)]
# keep only real part
Z_liq = np.real(Z_liq)
Z_liq

array([1.00022977, 1.00041138, 0.99952399, 1.0005407 , 0.99989551,
       1.00054582])

# Thermo

In [7]:
import numpy as np

In [8]:
from thermo import *
from scipy.constants import atm
pure_constants = ChemicalConstantsPackage.constants_from_IDs(
    ['co2', 'water', 'hydrogen', 'nitrogen', 'methanol'])

pseudos = ChemicalConstantsPackage(Tcs=[132.85], Pcs=[34.94*atm],
                                   omegas=[0.045], MWs=[28.01])
constants = pure_constants + pseudos

properties = PropertyCorrelationsPackage(constants=constants)

T = 600 + 273.15
P = 10e5

zs = np.array([0.2, 1e-10, 0.8, 1e-10, 1e-10, 1e-10])
zs = zs / np.sum(zs)
zs = list(zs)
eos_kwargs = dict(Tcs=constants.Tcs, Pcs=constants.Pcs, omegas=constants.omegas)#, kijs=kijs)

gas = CEOSGas(SRKMIX, eos_kwargs, HeatCapacityGases=properties.HeatCapacityGases)
liq = CEOSLiquid(SRKMIX, eos_kwargs, HeatCapacityGases=properties.HeatCapacityGases)

flashN = FlashVL(constants, properties, liquid=liq, gas=gas)

res = flashN.flash(T=T, P=P, zs=zs)

In [9]:
res.phase_count

1

In [10]:
type(res) is EquilibriumState 

True

In [11]:
res.VF

0.0